# Coffee Quality Prediction

In [11]:
import pandas as pd
import numpy as np

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
mpl.style.use('seaborn-v0_8-colorblind')
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, r2_score
import catboost as cb
from catboost import CatBoostRegressor, Pool

from sklearn.inspection import PartialDependenceDisplay, permutation_importance

import sys
import types
import IPython.display

# Create a mock submodule wrapper to mimic the old path
if "IPython.core.display" in sys.modules:
    sys.modules["IPython.core.display"].display = IPython.display.display
else:
    core_display = types.ModuleType("IPython.core.display")
    core_display.display = IPython.display.display
    core_display.HTML = IPython.display.HTML
    sys.modules["IPython.core.display"] = core_display
import lime, lime.lime_tabular
import shap

## Create Data (Initialisation Only)

In [12]:
# # New dataset with the new columns
# coffee = pd.DataFrame(columns=[
#     'Date',
#     'Method',
#     'Temperature',
#     'Humidity',
#     'Origin',
#     'Elevation',
#     'Region',
#     'Process',
#     'Varietal',
#     'DateRoast',
#     'DateOpen',
#     'Size',
#     'Grind',
#     'WaterTemp',
#     'Time',
#     'Output',
#     'Note',
#     'Score']
#     )

# with open('data/coffee_v2.csv', 'w') as f:
#     coffee.to_csv(f, index=False)

## Import Data

In [13]:
coffee = pd.read_csv('data/coffee_v2.csv')
coffee.head()

,Date,Method,Temperature,Humidity,Origin,Elevation,Region,Process,Varietal,DateRoast,DateOpen,Size,Grind,WaterTemp,Time,Output,Note,Score
0,2026-07-24,Filter,9,45,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,2026-07-18,53,16.5,98,180,300,NaN,5
1,2026-07-23,Filter,6,80,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,2026-07-18,53,16.5,99,180,303,NaN,4
2,2026-07-22,Filter,7,90,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,2026-07-18,53,16.7,98,180,300,NaN,4
3,2026-07-21,Filter,14,30,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,2026-07-18,53,16.4,96,180,300,NaN,4
4,2026-07-21,Filter,16,15,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,2026-07-18,53,16.5,98,180,300,NaN,5


In [14]:
coffee.dtypes

Date               str
Method             str
Temperature      int64
Humidity         int64
Origin             str
Elevation        int64
Region             str
Process            str
Varietal           str
DateRoast          str
DateOpen           str
Size             int64
Grind          float64
WaterTemp        int64
Time             int64
Output           int64
Note           float64
Score            int64
dtype: object

## Data Cleaning

In [15]:
coffee[['Date', 'DateRoast', 'DateOpen']] = coffee[['Date', 'DateRoast', 'DateOpen']].apply(pd.to_datetime)

## Feature Engineering

In [16]:
coffee['DayRoasted'] = (coffee['Date'] - coffee['DateRoast']).dt.days
coffee['DayOpened'] = (coffee['Date'] - coffee['DateOpen']).dt.days

coffee['ExtractedRatio'] = coffee['Output'] / coffee['Grind']

coffee.head()

,Date,Method,Temperature,Humidity,Origin,Elevation,Region,Process,Varietal,DateRoast,...,Size,Grind,WaterTemp,Time,Output,Note,Score,DayRoasted,DayOpened,ExtractedRatio
0,2026-07-24,Filter,9,45,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,...,53,16.5,98,180,300,NaN,5,43,6,18.181818
1,2026-07-23,Filter,6,80,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,...,53,16.5,99,180,303,NaN,4,42,5,18.363636
2,2026-07-22,Filter,7,90,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,...,53,16.7,98,180,300,NaN,4,41,4,17.964072
3,2026-07-21,Filter,14,30,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,...,53,16.4,96,180,300,NaN,4,40,3,18.292683
4,2026-07-21,Filter,16,15,Ethiopia,2050,Guji,Natural,Heirloom,2026-06-11,...,53,16.5,98,180,300,NaN,5,40,3,18.181818


## Export Cleaned Dataset

In [17]:
import sqlite3
conn = sqlite3.connect("database.db")
# Writes the dataframe directly to the SQL table
coffee.to_sql("coffee", conn, if_exists="append", index=False)
cur = conn.cursor()
cur.execute("SELECT * FROM coffee")
for row in cur.fetchall():
    print(row)
conn.close()

('2026-07-24 00:00:00', 'Filter', 9, 45, 'Ethiopia', 2050, 'Guji', 'Natural', 'Heirloom', '2026-06-11 00:00:00', '2026-07-18 00:00:00', 53, 16.5, 98, 180, 300, None, 5, 43, 6, 18.181818181818183)
('2026-07-23 00:00:00', 'Filter', 6, 80, 'Ethiopia', 2050, 'Guji', 'Natural', 'Heirloom', '2026-06-11 00:00:00', '2026-07-18 00:00:00', 53, 16.5, 99, 180, 303, None, 4, 42, 5, 18.363636363636363)
('2026-07-22 00:00:00', 'Filter', 7, 90, 'Ethiopia', 2050, 'Guji', 'Natural', 'Heirloom', '2026-06-11 00:00:00', '2026-07-18 00:00:00', 53, 16.7, 98, 180, 300, None, 4, 41, 4, 17.964071856287426)
('2026-07-21 00:00:00', 'Filter', 14, 30, 'Ethiopia', 2050, 'Guji', 'Natural', 'Heirloom', '2026-06-11 00:00:00', '2026-07-18 00:00:00', 53, 16.4, 96, 180, 300, None, 4, 40, 3, 18.29268292682927)
('2026-07-21 00:00:00', 'Filter', 16, 15, 'Ethiopia', 2050, 'Guji', 'Natural', 'Heirloom', '2026-06-11 00:00:00', '2026-07-18 00:00:00', 53, 16.5, 98, 180, 300, None, 5, 40, 3, 18.181818181818183)
